## Requirements and Running Instructions

### Required packages
- Python 3
- pandas
- numpy
- scikit-learn
- xgboost
- matplotlib
- seaborn

Install the required packages by running this in a notebook code cell:

    %pip install pandas numpy scikit-learn xgboost matplotlib seaborn

The modules `collections`, `os`, and `random` are included with Python.

### Required data
Place `cafeteriasimulationdata.csv` in the location specified by the
`pd.read_csv()` call. This file contains the data used to train the
ML demand predictor.

The notebook generates the 50 evaluation lunches automatically.

### Reproducibility
A random seed of 42 is used to reproduce the generated lunches.
All three strategies are evaluated using the same student arrivals.

---

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
import random
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

---
### Loading the data that will be used to train the Machine learning model 

In [2]:
simulation_data = pd.read_csv('../CafeteriaSimulation/cafeteriasimulationdata.csv')
simulation_data.head(5)

,minute,rate,arrivals,last_5_min,last_10_min,type,next_10_min,lunch_id
0,1,3,4,4.0,4.0,slow,35.0,slow1
1,2,3,4,8.0,8.0,slow,32.0,slow1
2,3,3,5,13.0,13.0,slow,33.0,slow1
3,4,3,1,14.0,14.0,slow,34.0,slow1
4,5,3,7,21.0,21.0,slow,32.0,slow1


### Training a model 

In [3]:
model = XGBRegressor(objective = 'reg:squarederror', n_estimators = 200, max_depth = 3, learning_rate = 0.05, subsample= 0.8, colsample_bytree = 1.0, random_state =42)
#model = DecisionTreeRegressor(random_state=42) MAE Score : 12
#model = LogisticRegression(random_state=42) MAE Score 13

### Splitting the data into train test split in order to train and test the data 

In [4]:
simulation_data_c = simulation_data.copy()
## Since there are 100 samples of each lunch_id we will grab only one of each 
lunch_info = simulation_data_c[['lunch_id', 'type']].drop_duplicates()
train_ids, test_ids = train_test_split(lunch_info['lunch_id'], test_size =0.2, stratify=lunch_info["type"], random_state=42)
train = simulation_data_c[simulation_data_c['lunch_id'].isin(train_ids)]
test = simulation_data_c[simulation_data_c['lunch_id'].isin(test_ids)]

In [5]:
### X_train,y_train, X_test, y_test
features = ['minute', 'last_5_min', 'last_10_min']
X_train = train[features]
y_train = train['next_10_min']
X_test = test[features]
y_test = test['next_10_min']


In [6]:
lunch_info.head(10)


,lunch_id,type
0,slow1,slow
110,slow2,slow
220,slow3,slow
330,slow4,slow
440,slow5,slow
550,slow6,slow
660,slow7,slow
770,slow8,slow
880,slow9,slow
990,slow10,slow


In [7]:
### Overview of X_train 
X_train.tail(5)

,minute,last_5_min,last_10_min
32995,106,34.0,61.0
32996,107,39.0,63.0
32997,108,37.0,63.0
32998,109,30.0,65.0
32999,110,35.0,69.0


### Predicting the students arrival per minute 

In [8]:
### training the model 
model.fit(X_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

### The machine learning model has been created
---
### Creating simulation to run all the cooking strategies

In [9]:
from collections import deque
def simulation(lunch_data, ml_simulation = False, reactive_simulation = False, fixed_simulation = False): ## Using the simulation function to generate simulations
    waiting_times = [] ## Calculating the total time student had to wait 
    lunch_duration = 120 ## The lunch will run for 120 minutes 
    cooking_time = 10 ## Total time required to cook the food 
    batch_size = 65 ## Portions produced per batch 
    available_portions = 65 ## Food currently ready to serve 
    cooking_finish_time = None ## No batch is getting cooked currently 
    scheduled_interval = 10
    reactive_threshold = 35 ## For reactive simulation: Start cooking when the available portions < reactive threshold
    queue = deque() ## Starting unique queue for each of the input
    records = []
    predicted_demand = None
    batches_started = 0
    if sum([ml_simulation, reactive_simulation, fixed_simulation]) != 1:
        raise ValueError("Choose exactly one strategy.") ### Only choose one simulation 
    for _ , row in lunch_data.iterrows():
        minute = int(row['minutes']) ## Minute count 
        arrivals = int(row['arriving_students']) ## Total number of students arriving in that minute 
        ### Adding the food 
        if cooking_finish_time is not None: ## If something is getting cooked
            if minute >= cooking_finish_time: ## If the minute or time
                available_portions += batch_size ## We add the total food to the batch size 
                cooking_finish_time = None 
        queue.extend([minute] * arrivals) ## Extend queue by each people arrivals per minute. if 3 people come in minute 10, the queue will be extended by [10,10,10]
        food_served = min(available_portions, len(queue)) ## The food will be served if there is available 
        for _ in range(food_served): ## Looping through the ones who got served
            arrival_minute = queue.popleft() ## Finding the minute they came in
            waiting_times.append(minute-arrival_minute) ## Minute now and the time they were served and appending the waiting time to the list 
        available_portions -= food_served ## Decreasing the available_portions with the food served
        if ml_simulation == True: ## If cooking strategy is ml simulation 
            predicted_demand = (max(0, float(row['predicted_next_10_min']))) ## total demand predicted by machine learning model 
            if (cooking_finish_time is None and available_portions < len(queue) + predicted_demand and minute + cooking_time <= lunch_duration ): ## Setting up the conditions 
                cooking_finish_time = minute + cooking_time ## If the minute is 30 and the food is need at minute 30 it will take 10 minutes to make the food meaning the fodd will be available at 30 + 10 = 40 
                batches_started += 1 ## Total number of batches cooked 
        if reactive_simulation == True: 
            if (cooking_finish_time is None and available_portions < reactive_threshold and minute + cooking_time <= lunch_duration ):
                cooking_finish_time = minute + cooking_time ## If the minute is 30 and the food is need at minute 30 it will take 10 minutes to make the food meaning the fodd will be available at 30 + 10 = 40 
                batches_started += 1 ## Total number of batches cooked 
        if fixed_simulation == True: 
            if (cooking_finish_time is None and minute % scheduled_interval == 0 and minute + cooking_time <= lunch_duration ):
                cooking_finish_time = minute + cooking_time ## If the minute is 30 and the food is need at minute 30 it will take 10 minutes to make the food meaning the fodd will be available at 30 + 10 = 40 
                batches_started += 1 ## Total number of batches cooked 

        records.append({
            "minute": minute,
            "arrivals": arrivals,
            "predicted_demand": predicted_demand,
            "available_portions": available_portions,
            "students_waiting": len(queue),
            "cooking": cooking_finish_time is not None,
        })

    simulation_result = pd.DataFrame(records)
    return simulation_result, waiting_times

## Evaluating simulation on 50 different datasets 

### Creating 50 samples of datasets to evaluate the model 

In [10]:
def dataset_creater(number_of_times):
    datas = [] ## To use for machine learning simulation only
    rng = np.random.default_rng(42)
    for i in range(number_of_times):
        data = pd.DataFrame({
        "minutes": np.arange(1,121),
        "rate": 1
        })

    ## Here we apply the rate in which the data was created Slow Rush : 1, Normal Rush: 2, and Busy Rush:5
        data.loc[data['minutes'].between(1,30), "rate"] = rng.choice([2, 5])
        data.loc[data['minutes'].between(31,60), "rate"] = rng.choice([2, 5])
        data.loc[data['minutes'].between(61,90), "rate"] = rng.choice([2, 5])
        data.loc[data['minutes'].between(91,120), "rate"] = rng.choice([2, 5])
        data['arriving_students'] = rng.poisson(data['rate'])

        data["last_5_min"] = (data["arriving_students"].rolling(5, min_periods=1).sum()) ## Cumulative sum for first 5
        data["last_10_min"] = (data["arriving_students"].rolling(10, min_periods=1).sum()) ## Cumulative sum for first 10
        datas.append(data)
    return datas 


In [11]:
simulationdatas = dataset_creater(50) ##This will also create the 50 datasets that will be used in the fixedschedule and reactive simulation

### Model_prediction: predicts the next 10 min data which will be used for machine learning simulation

In [12]:
## Predicting each model's value using the function 
def model_prediciton(model, dataset, duration = 120 ):
    predicted_outputs = [] ## This store the predicted output 
    for i in range(duration): ## Range of lunch duration which is 120 minutes 
        every_min_data = dataset[dataset['minutes'] == i+1] ## Each minute
        minutes_of_arrival = every_min_data['minutes'].iloc[0] ## Minute of arrival 
        last_5_mins = every_min_data['last_5_min'].iloc[0] ## Last 5 minutes data
        last_10_mins = every_min_data['last_10_min'].iloc[0] ## Last 10 minutes data
        current_input = pd.DataFrame({"minute": [minutes_of_arrival], "last_5_min":[last_5_mins], "last_10_min":[last_10_mins]}) ## Current input 
        predicted_output = model.predict(current_input) ## Preidcted output for each dataset 
        predicted_outputs.append(predicted_output.item()) ## Appending in the list to make them column later

    return predicted_outputs ## Returning the value
    

In [13]:
for_ml_model_only = dataset_creater(50) 

### Predicting next 10 minute data for each dataset

In [14]:
for each_dataset in for_ml_model_only: 
    predicted_next_10_minutes =  model_prediciton(model, each_dataset) ## predicting the next 10 minutes of the data
    each_dataset['predicted_next_10_min'] = predicted_next_10_minutes ## Next 10 minutes data added to dataframe


### Running simulation through 50 datasets for each simulation 

In [15]:
def running_simulation(datasets, ml_simulation = False, reactive_simulation = False, fixed_simulation = False):
    rows = []
    for d in datasets:
        result, waits = simulation(d, ml_simulation = ml_simulation, reactive_simulation = reactive_simulation, fixed_simulation = fixed_simulation)
        rows.append({
            "total_arrivals": d["arriving_students"].sum(),
            "leftover": result["available_portions"].iloc[-1],
            "waiting_end": result["students_waiting"].iloc[-1],
            "mean_wait": np.mean(waits),
            "p95_wait": np.percentile(waits, 95),
            "max_queue": result["students_waiting"].max(),
        })

    summary = pd.DataFrame(rows)
    return summary

    # plt.scatter(summary["total_arrivals"], summary["leftover"], label="Leftover")
    # plt.scatter(summary["total_arrivals"], summary["waiting_end"], label="Waiting at end")
    # plt.xlabel("Total arrivals"); plt.legend(); plt.show()

### Simulation 1: Machine Learning Simulation 

In [16]:
machine_learning_simulation_results = running_simulation(for_ml_model_only, True, False, False)

In [17]:
## Summary of machine learning results 
machine_learning_simulation_results.to_csv('../CafeteriaSimulation/results/machinelearning_results.csv', index = False)
machine_learning_simulation_results.tail(10)

,total_arrivals,leftover,waiting_end,mean_wait,p95_wait,max_queue
40,507,13,0,0.428008,3.0,27
41,511,9,0,0.391389,3.0,35
42,296,29,0,0.000000,0.0,0
43,402,53,0,0.273632,3.0,26
44,515,5,0,0.077670,1.0,12
45,321,4,0,0.274143,2.0,19
46,599,51,0,0.101836,1.0,17
47,444,11,0,0.027027,0.0,6
48,529,0,9,0.101923,1.0,15
49,629,21,0,0.222576,2.0,25


### Simulation 2: Reactive Threshold Simulation 

In [18]:
reactive_simulation_results = running_simulation(simulationdatas, False, True, False)

In [19]:
reactive_simulation_results.to_csv('../CafeteriaSimulation/results/reactivethreshold_result.csv', index = False)
reactive_simulation_results.head(10)

,total_arrivals,leftover,waiting_end,mean_wait,p95_wait,max_queue
0,408,47,0,0.370098,3.0,25
1,505,15,0,0.461386,3.0,22
2,252,73,0,0.000000,0.0,0
3,416,39,0,0.211538,2.0,19
4,620,30,0,0.490323,3.0,24
5,383,7,0,0.195822,2.0,16
6,428,27,0,0.301402,2.0,22
7,309,81,0,0.346278,3.0,20
8,461,0,6,0.184615,2.0,15
9,487,33,0,0.422998,3.0,22


### Simulation 3: Fixed Schedule Simulation 

In [20]:
fixed_schedule_simulation = running_simulation(simulationdatas, False, False, True)

In [21]:
fixed_schedule_simulation.to_csv('../CafeteriaSimulation/results/fixedschedule_result.csv', index = False)
fixed_schedule_simulation.head(10)

,total_arrivals,leftover,waiting_end,mean_wait,p95_wait,max_queue
0,408,372,0,0.000000,0.00,0
1,505,275,0,0.330693,2.80,31
2,252,528,0,0.000000,0.00,0
3,416,364,0,0.000000,0.00,0
4,620,160,0,0.116129,0.05,19
5,383,397,0,0.000000,0.00,0
6,428,352,0,0.000000,0.00,0
7,309,471,0,0.000000,0.00,0
8,461,319,0,0.000000,0.00,0
9,487,293,0,0.318275,2.70,31
